In [ ]:
# Cell 0.1: Cài đặt thư viện cần thiết (nếu chưa có)
!pip install vnstock keras-tuner tensorflow scikit-learn matplotlib pandas numpy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.0 MB/s eta 0:00:00


In [ ]:
# Cell 0.2: Import các thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
from vnstock import Vnstock  # Thư viện lấy dữ liệu chứng khoán Việt Nam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Cấu hình hiển thị
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid') # Chọn style cho plot
pd.options.display.float_format = '{:,.2f}'.format # Định dạng số float

In [ ]:
# Cell 0.3: Các tham số cấu hình
STOCK_SYMBOL = 'HPG'
SOURCE = 'VCI' # Nguồn dữ liệu (vd: 'SSI', 'VCI', 'TCBS')
START_DATE = '2015-04-30' # Bắt đầu lấy dữ liệu từ ngày này (có thể lấy dài hơn để có đủ dữ liệu cho features)
END_DATE = '2025-04-30'   # Kết thúc lấy dữ liệu (có thể dùng datetime.today().strftime('%Y-%m-%d'))
VAL_DATE_START = '2024-01-01' # Ngày bắt đầu của tập validation
TEST_DATE_START = '2025-04-10' # Ngày bắt đầu của tập test

SEQUENCE_LENGTH = 60     # Độ dài chuỗi input cho LSTM (số ngày nhìn lại)
FEATURES = ['close', 'open', 'high', 'low', 'volume',
            'MA5', 'MA20', 'RSI', 'MACD', 'Signal_Line',
            'Price_Change', 'Volatility'] # Các features sử dụng

# Tham số mô hình LSTM
LSTM_UNITS = 128
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.00001
EPOCHS = 100
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 15

MODEL_FILENAME = 'lstm_stock_model_vhc.h5'

In [ ]:
from google.colab import files  # <-- Thêm dòng này nếu đang dùng Colab

print(f"Đang tải dữ liệu cổ phiếu {STOCK_SYMBOL} từ {START_DATE} đến {END_DATE}...")
try:
    stock_data_loader = Vnstock().stock(symbol=STOCK_SYMBOL, source=SOURCE)
    df_raw = stock_data_loader.quote.history(start=START_DATE, end=END_DATE, interval='1D')
    print("Tải dữ liệu thành công!")
    print(f"Số dòng dữ liệu gốc: {len(df_raw)}")
    display(df_raw.head())
    display(df_raw.tail())

    # 🔽 Lưu và tải file về
    file_name = f"{STOCK_SYMBOL}_history.csv"
    df_raw.to_csv(file_name, index=False)
    files.download(file_name)  # ⬅️ Tải về máy

except Exception as e:
    print(f"Lỗi khi tải dữ liệu: {e}")
    df_raw = pd.DataFrame()  # Tạo dataframe rỗng nếu lỗi


Đang tải dữ liệu cổ phiếu HPG từ 2015-04-30 đến 2025-04-30...
Tải dữ liệu thành công!
Số dòng dữ liệu gốc: 2502


,time,open,high,low,close,volume
0,2015-05-04,3.48,3.48,3.35,3.37,1045680
1,2015-05-05,3.37,3.48,3.37,3.44,767180
2,2015-05-06,3.45,3.47,3.39,3.39,993740
3,2015-05-07,3.39,3.43,3.39,3.41,540380
4,2015-05-08,3.45,3.53,3.37,3.37,565420


,time,open,high,low,close,volume
2497,2025-04-23,25.50,25.70,25.35,25.55,22232700
2498,2025-04-24,25.70,25.70,25.20,25.60,17212000
2499,2025-04-25,25.45,25.70,25.25,25.70,24849000
2500,2025-04-28,25.70,25.70,25.35,25.65,9757000
2501,2025-04-29,25.65,25.70,25.40,25.50,14467400


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 1.2: Kiểm tra dữ liệu ban đầu
if not df_raw.empty:
    print("\nThông tin dữ liệu (Info):")
    df_raw.info()

    print("\nThống kê mô tả (Describe):")
    display(df_raw.describe())

    print("\nKiểm tra giá trị thiếu (Missing Values):")
    print(df_raw.isnull().sum())
else:
    print("Không có dữ liệu để xử lý.")


Thông tin dữ liệu (Info):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2502 entries, 0 to 2501
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   time    2502 non-null   datetime64[ns]
 1   open    2502 non-null   float64       
 2   high    2502 non-null   float64       
 3   low     2502 non-null   float64       
 4   close   2502 non-null   float64       
 5   volume  2502 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 117.4 KB

Thống kê mô tả (Describe):


,time,open,high,low,close,volume
count,2502,"2,502.00","2,502.00","2,502.00","2,502.00","2,502.00"
mean,2020-04-25 20:52:56.978417152,71.41,72.06,70.78,71.39,"1,825,185.05"
min,2015-05-04 00:00:00,32.87,33.19,32.55,33.19,0.00
25%,2017-10-24 06:00:00,63.99,64.47,63.47,63.89,"692,072.75"
50%,2020-04-27 12:00:00,70.93,71.70,70.26,70.92,"1,319,855.00"
75%,2022-10-24 18:00:00,79.39,80.13,78.87,79.49,"2,514,050.00"
max,2025-04-29 00:00:00,110.65,110.91,109.93,110.81,"21,167,413.00"
std,NaN,13.96,14.06,13.83,13.96,"1,672,389.50"



Kiểm tra giá trị thiếu (Missing Values):
time      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64
